# Actividad 00 - Pesos geograficos historicos (crop-aware, congelados)

**Fase 2 v2 - reconstruccion aprobada 2026-09-07**

Genera el ponderador espacial que sustituye a la agregacion anterior, la cual
usaba la produccion provincial del **mismo mes t** y por tanto incurria en
look-ahead operacional respecto del target.

| Decision | Contenido |
|---|---|
| **D14** | Base = `verde_actual_ha` (superficie en produccion activa). NO se usa produccion. |
| **D15** | Resumen = **media** sobre **TRAIN 2016-01..2023-12**. Val 2024 y test 2025 excluidos. |
| **D16** | **Dos vectores independientes** (Sutil / Dulce). PROHIBIDO el vector combinado. |
| **D17** | Sin exclusion de provincias, sin piso minimo, sin suavizado. |
| **D19** | SHA-256 de fuente y artefacto (**convencion nueva de v2**). |
| **D21** | Los CSV provinciales se tratan como **inputs congelados**; no se reconstruye Fase 1. |

**Salidas:** `resultados_v2_final/pesos/pesos_geograficos_train.csv` y su metadata JSON.

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 240); pd.set_option('display.max_columns', None)

while not os.path.exists('v2_reentrenamiento/data/processed'):
    os.chdir('..')
RAIZ = os.getcwd()
sys.path.insert(0, os.path.join(RAIZ, 'v2_reentrenamiento', 'src'))
print('Raiz del proyecto:', RAIZ)

PROC      = 'v2_reentrenamiento/data/processed'
INTERIM   = 'v2_reentrenamiento/data/interim'
RAW       = 'v2_reentrenamiento/data/raw'
OUT_R2    = 'v2_reentrenamiento/resultados_v2_final'
PESOS_CSV = f'{OUT_R2}/pesos/pesos_geograficos_train.csv'
CULTIVARES = ['sutil', 'dulce']

In [ ]:
from features.pesos_geograficos import PesosGeograficos, VentanaTrain
from features.hashing import Hasher

VENTANA = VentanaTrain(2016, 1, 2023, 12)
print('Ventana de ajuste (D15):', VENTANA.inicio, '..', VENTANA.fin)
print('Variable base (D14)    : verde_actual_ha')
print('Resumen temporal (D15) : media')

## 1. Carga de los paneles provinciales (inputs congelados - D21)

In [ ]:
paneles, fuentes = {}, {}
for c in CULTIVARES:
    ruta = f'{INTERIM}/limon_{c}_provincia.csv'
    paneles[c] = pd.read_csv(ruta, encoding='utf-8-sig')
    fuentes[c] = Hasher.describir(ruta)
    d = paneles[c]
    print(f'{c.upper():6s} {d.shape}  {d["año"].min()}-{d["año"].max()}  '
          f'provincias={d.groupby(["departamento","provincia"]).ngroups}  NaN={int(d.isna().sum().sum())}')
    print(f'       sha256 = {fuentes[c]["sha256"]}')

## 2. Ajuste de los pesos - un vector independiente por cultivar (D16)

In [ ]:
pesos = {}
for c in CULTIVARES:
    pesos[c] = PesosGeograficos(cultivar=c, ventana=VENTANA).ajustar(paneles[c])
    r = pesos[c].resumen_concentracion()
    print(f'{c.upper():6s} n={r["n_provincias"]:4d}  peso>0={r["n_provincias_peso_positivo"]:4d}  '
          f'sum={pesos[c].tabla["peso"].sum():.12f}  HHI={r["hhi"]:.4f}  N_eff={r["n_efectivo"]:.2f}  '
          f'top5={r["top5"]:.4f}  n50={r["n_para_50pct"]} n80={r["n_para_80pct"]} n95={r["n_para_95pct"]}')

### Checks obligatorios por cultivar

In [ ]:
for c in CULTIVARES:
    t = pesos[c].tabla
    s = float(t['peso'].sum())
    assert abs(s - 1.0) <= 1e-9, f'{c}: sum(peso)={s}'
    assert (t['peso'] >= 0).all(), f'{c}: pesos negativos'
    assert not t.duplicated(['cultivar', 'departamento', 'provincia']).any(), f'{c}: duplicados'
    assert (t['periodo_fin'] == '2023-12').all(), f'{c}: periodo_fin != 2023-12'
    clave = paneles[c]['año'] * 100 + paneles[c]['mes']
    usadas = paneles[c][(clave >= 201601) & (clave <= 202312)]
    maxk = int((usadas['año'] * 100 + usadas['mes']).max())
    assert maxk <= 202312
    print(f'{c.upper():6s} OK | sum(peso)-1 = {s-1:+.2e} | filas train usadas = {len(usadas)} | max(anio-mes) = {maxk}')

### Top-10 por cultivar

In [ ]:
for c in CULTIVARES:
    print(f'--- {c.upper()} ---')
    print(pesos[c].tabla.head(10)[['ranking','departamento','provincia','valor_historico','peso','peso_acumulado']]
          .to_string(index=False))
    print()

## 3. Persistencia + metadata con SHA-256 (D19)

In [ ]:
from datetime import datetime

tabla = PesosGeograficos.concatenar([pesos[c] for c in CULTIVARES])
os.makedirs(f'{OUT_R2}/pesos', exist_ok=True)
tabla.to_csv(PESOS_CSV, index=False, encoding='utf-8-sig')
print('Guardado:', PESOS_CSV, tabla.shape)

meta = {
    'artefacto': 'pesos_geograficos_train.csv',
    'fecha_generacion': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'decisiones': ['D14 base=verde_actual_ha',
                   'D15 media sobre TRAIN 2016-01..2023-12',
                   'D16 vectores independientes por cultivar (combinado PROHIBIDO)',
                   'D17 sin exclusion / sin piso minimo / sin suavizado',
                   'D19 SHA-256 (convencion nueva de v2)',
                   'D21 interim MIDAGRI tratado como input congelado'],
    'variable_base': 'verde_actual_ha', 'unidad': 'ha', 'resumen_temporal': 'media',
    'columnas_utilizadas': ['anio', 'mes', 'departamento', 'provincia', 'verde_actual_ha'],
    'split_utilizado': {'train': '2016-01..2023-12', 'val_excluida': '2024-01..2024-12',
                        'test_excluida': '2025-01..2025-12'},
    'procedencia': {
        'origen_primario': 'data/raw/midagri/sisagri_2015_2026.xlsx',
        'nota_D21': ('La cadena xlsx -> distrito -> provincia NO es reproducible: no existe codigo '
                     'en el repositorio que la genere. Los CSV provinciales se tratan como inputs '
                     'congelados y se registran por SHA-256. La reconstruccion completa de Fase 1 '
                     'queda fuera del alcance de esta correccion.')},
    'fuentes': fuentes,
    'por_cultivar': {c: {**pesos[c].resumen_concentracion(),
                         'suma_pesos': float(pesos[c].tabla['peso'].sum())} for c in CULTIVARES},
    'artefacto_generado': Hasher.describir(PESOS_CSV),
}
ruta_meta = f'{OUT_R2}/pesos/pesos_geograficos_train_meta.json'
json.dump(meta, open(ruta_meta, 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
print('Guardado:', ruta_meta)
print('sha256 pesos =', meta['artefacto_generado']['sha256'])